# fact_video_restaurant_mention_poc (factless bridge)

Grain: one video × one restaurant (`video_id`, `restaurant_id`). `mention_weight = 1 / restaurants in the video` (sums to 1 per video). Restaurants not geocoded yet share one `restaurant_id = 'unknown'` row per video, keeping their share (`mention_count` = how many).

`restaurant_id` is built the same way as in `restaurants.ipynb`: `md5('place:' || place_id)` when Google matched the place, `md5('name:' || name || '|' || district)` for an approximate OpenStreetMap point, else unknown.

**Incremental:** a video is rebuilt as a whole (its weights change together) when it was extracted in the window, when one of its restaurants was geocoded in the window, or when it still has an `unknown` row. Unlike poc `upsert`, the old rows of those videos that are no longer in the source are then **deleted** (e.g. the `unknown` row once the restaurant is geocoded); rows of other videos are not touched.

Run by `facts_poc` (task `gold_facts_poc`), after `gold_dims_scd1`. Keys come from the `*_scd1` dims (same `xxhash64` keys as the live dims); no match → −1. Test table: the live fact and the agg tables are unchanged.

In [ ]:
%run ./_udfs

In [ ]:
%run ./_scd

# Set up environment

In [ ]:
for layer in ("silver", "gold"):
    dbutils.widgets.text(layer, f"prod.{layer}")
dbutils.widgets.dropdown("load_type", "incremental", ["incremental", "full"])
dbutils.widgets.text("from_date", "")   # yyyy-MM-dd, Vietnam day, inclusive; empty = yesterday
dbutils.widgets.text("to_date", "")     # exclusive; empty = tomorrow

silver = dbutils.widgets.get("silver")
gold = dbutils.widgets.get("gold")
load_type = dbutils.widgets.get("load_type")
from_date, to_date = get_incremental_dates(dbutils.widgets.get("from_date"), dbutils.widgets.get("to_date"))
print(f"load_type = {load_type}, window = [{from_date}, {to_date})")

# Read Source Table

In [ ]:
path = f"{gold}.fact_video_restaurant_mention_poc"
mentions = spark.table(f"{silver}.stg_restaurant_mentions")
geocode = (spark.table(f"{silver}.stg_restaurant_geocode")
           .filter(F.col("match_status").isin("matched", "places_match", "osm_approx")))
dim_video = spark.table(f"{gold}.dim_video_scd1").filter("video_key <> -1")

## Normalized district (same rule as `stg_restaurant_geocode`)

`Q.7` / `Quận 07` → `quan 7`, `TP. Thủ Đức` → `thu duc`.

In [ ]:
district_norm = F.expr(r"""
    nullif(trim(regexp_replace(regexp_replace(regexp_replace(
        lower(unaccent_vi(coalesce(district, ''))),
        '^(q\\.?\\s*|quan\\s+)0*(\\d+)$', 'quan $2'),
        '^(tp\\.?|thanh pho)\\s+', ''),
        '\\s+', ' ')), '')""")
mentions = mentions.withColumn("district_norm", district_norm)

## Incremental: videos to rebuild

In [ ]:
if load_type == "incremental":
    extracted_day = F.to_date(F.from_utc_timestamp("extracted_at", TZ))
    geocoded_day = F.to_date(F.from_utc_timestamp("geocoded_at", TZ))

    extracted_in_window = mentions.filter((extracted_day >= from_date) & (extracted_day < to_date)).select("video_id")
    geocoded_in_window = (mentions.alias("m")
        .join(geocode.filter((geocoded_day >= from_date) & (geocoded_day < to_date)).alias("g"),
              on=(F.col("m.restaurant_name_normalized") == F.col("g.restaurant_name_normalized"))
                 & F.col("m.district_norm").eqNullSafe(F.col("g.district_norm")))
        .select("m.video_id"))
    changed_videos = extracted_in_window.union(geocoded_in_window)
    if spark.catalog.tableExists(path):
        changed_videos = changed_videos.union(spark.table(path).filter("restaurant_id = 'unknown'").select("video_id"))
    changed_videos = changed_videos.distinct()
    mentions = mentions.join(changed_videos, "video_id", "left_semi")

# Transformation

## restaurant_id of each mention

In [ ]:
mention_restaurant = (
    mentions.alias("m")
    .join(geocode.alias("g"),
          on=(F.col("m.restaurant_name_normalized") == F.col("g.restaurant_name_normalized"))
             & F.col("m.district_norm").eqNullSafe(F.col("g.district_norm")),
          how="left")
    .select(
        F.col("m.video_id").alias("video_id"),
        F.when(F.col("g.place_id").isNotNull(), F.md5(F.concat(F.lit("place:"), F.col("g.place_id"))))
         .when(F.col("g.match_status") == "osm_approx",
               F.md5(F.concat(F.lit("name:"), F.col("m.restaurant_name_normalized"), F.lit("|"),
                              F.coalesce(F.col("m.district_norm"), F.lit("")))))
         .alias("restaurant_id"),
        F.col("m.mention_source").alias("mention_source"),
        F.concat(F.col("m.restaurant_name_normalized"), F.lit("|"),
                 F.coalesce(F.col("m.district_norm"), F.lit(""))).alias("name_key"),
    )
    .distinct()
)

## One row per video × restaurant

Unmatched mentions collapse into one `unknown` row per video; `n` = restaurants it stands for.

In [ ]:
from pyspark.sql.window import Window

per_restaurant = (
    mention_restaurant
    .groupBy("video_id", "restaurant_id")
    .agg(F.max("mention_source").alias("mention_source"),
         F.when(F.col("restaurant_id").isNull(), F.countDistinct("name_key")).otherwise(F.lit(1)).alias("n"))
    .withColumn("restaurant_id", F.coalesce(F.col("restaurant_id"), F.lit("unknown")))
)

## Join dims and select columns

In [ ]:
per_video = Window.partitionBy(F.col("m.video_id"))   # mention_weight = n / restaurants in the video

result_df = (
    per_restaurant.alias("m")
    .join(dim_video.alias("video"), on=F.col("m.video_id") == F.col("video.video_id"), how="left")
    .select(
        F.col("m.video_id").cast("string").alias("video_id"),
        F.col("m.restaurant_id").cast("string").alias("restaurant_id"),
        F.coalesce(F.col("video.video_key"), F.lit(-1)).cast("bigint").alias("video_key"),
        F.col("m.mention_source").cast("string").alias("mention_source"),
        F.least(F.col("m.n"), F.lit(127)).cast("tinyint").alias("mention_count"),
        (F.col("m.n") / F.sum("m.n").over(per_video)).cast("double").alias("mention_weight"),
    )
)

## Add metadata

In [ ]:
insert_date = datetime.now()
etl_by = "Databricks"
result_df = result_df.withColumn("etl_date", F.lit(insert_date)).withColumn("etl_by", F.lit(etl_by))

## Add checksums

In [ ]:
primary_key = ["video_id", "restaurant_id"]
result_df = add_checksums(result_df, primary_key)

## Create table

In [ ]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {path} (
        video_id STRING, restaurant_id STRING, video_key BIGINT, mention_source STRING,
        mention_count TINYINT, mention_weight DOUBLE,
        etl_date TIMESTAMP, etl_by STRING, primary_checksum STRING, record_checksum STRING
    ) USING DELTA
    COMMENT 'Factless bridge (poc checksum method): 1 video x 1 restaurant (unknown = not geocoded yet). mention_weight = 1 / restaurants in the video (sums to 1).'
""")
result_df = result_df.select(*spark.table(path).columns)

# Write to Table

**full**: overwrite. **incremental**: upsert (MERGE on `primary_checksum`: update when `record_checksum` changed, insert new rows), **then delete** the rows of the rebuilt videos that are no longer in the source (second MERGE, poc variant "delete then write").

In [ ]:
if load_type == "full":
    full_refresh(result_df, path)
else:
    result_df.createOrReplaceTempView("source")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING source AS s
        ON t.primary_checksum = s.primary_checksum
        WHEN MATCHED AND t.record_checksum <> s.record_checksum THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

    # rows of the rebuilt videos that the source no longer has
    to_delete = (spark.table(path)
                 .join(changed_videos, "video_id", "left_semi")
                 .join(result_df.select("primary_checksum"), "primary_checksum", "left_anti")
                 .select("primary_checksum"))
    to_delete.createOrReplaceTempView("to_delete")
    spark.sql(f"""
        MERGE INTO {path} AS t
        USING to_delete AS d
        ON t.primary_checksum = d.primary_checksum
        WHEN MATCHED THEN DELETE
    """)

## Check: one row per grain

In [ ]:
n, grain = spark.sql(f"SELECT COUNT(*), COUNT(DISTINCT primary_checksum) FROM {path}").first()
assert n == grain, f"{path}: {n - grain} duplicate rows on (video_id, restaurant_id)"
last = spark.sql(f"DESCRIBE HISTORY {path} LIMIT 1").select("operation", "operationMetrics").first()
print(f"✅ {path}: {n} rows ·", last["operation"], dict(last["operationMetrics"] or {}))